<a href="https://colab.research.google.com/github/cepulmano/ai-healthcare-medicine/blob/main/day3/MedicalSpecialtyClassification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install evaluate clean-text

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.3 MB/s eta 0:00:00


In [4]:
import re
import torch
import numpy as np
import pandas as pd
import evaluate
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, pipeline
from cleantext import clean

In [18]:
# HARDWARE CHECK & OPTIMIZATION
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cpu


In [20]:
# LOAD THE DATASET
df = pd.read_csv("https://raw.githubusercontent.com/cepulmano/ai-healthcare-medicine/refs/heads/main/day3/mtsamples.csv")
df.head()

,index,description,medical_specialty,sample_name,transcription,keywords
0,18,Fertile male with completed family. Elective...,Urology,Vasectomy - 4,"PROCEDURE: , Elective male sterilization via b...","urology, sterilization, vas, fertile male, bil..."
1,20,Whole body radionuclide bone scan due to pros...,Urology,Whole Body Radionuclide Bone Scan,"INDICATION:, Prostate Cancer.,TECHNIQUE:, 3....","urology, prostate cancer, technetium, whole bo..."
2,22,Normal vasectomy,Urology,Vasectomy - 1,"DESCRIPTION:, The patient was placed in the s...","urology, vasectomy, allis clamp, catgut, hemoc..."
3,23,Voluntary sterility. Bilateral vasectomy. T...,Urology,Vasectomy,"PREOPERATIVE DIAGNOSIS: , Voluntary sterility....","urology, hemiscrotum, bilateral vasectomy, vol..."
4,24,Blood in urine - Transitional cell cancer of ...,Urology,Urology Consut - 1,"CHIEF COMPLAINT:,",NaN


In [21]:
# Map medical specialty string labels to unique numbers automatically
df["medical_specialty"] = df["medical_specialty"].astype("category")
df["label"] = df["medical_specialty"].cat.codes

# Create dictionary maps to translate numbers back to specialty names during prediction
id2label = dict(enumerate(df["medical_specialty"].cat.categories))
label2id = {v: k for k, v in id2label.items()}

num_classes = len(id2label)

print(f"Detected {num_classes} distinct medical specialties to classify.")

# Format columns for Hugging Face
df = df.rename(columns={"transcription": "text"})
df_clean = df[["text", "label"]]

Detected 2 distinct medical specialties to classify.


In [22]:
from sklearn.utils.class_weight import compute_class_weight

# Calculate class weights based on label frequencies
labels = df_clean["label"].values
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(labels),
    y=labels
)

# Convert to a PyTorch tensor and send to your GPU/CPU device
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
print(f"Computed weights for {len(class_weights_tensor)} classes.")

Computed weights for 2 classes.


In [23]:
# Convert to Dataset and split into Train (80%) and Test/Validation (20%)
base_dataset = Dataset.from_pandas(df_clean)
dataset_splits = base_dataset.train_test_split(test_size=0.20, seed=42)
dataset = DatasetDict({
    "train": dataset_splits["train"],
    "validation": dataset_splits["test"]
})

In [24]:
# MEDICAL TEXT CLEANING FUNCTION
def clean_medical_text(text):
    if not isinstance(text, str):
        return ""
    # Strip common clinical formatting characters or punctuation noise if needed
    text = re.sub(r'<[^>]+>', '', text)
    return clean(
        text, fix_unicode=True, to_ascii=True, lower=True,
        no_urls=True, no_emails=True, no_phone_numbers=True, lang="en"
    )

print("Cleaning text transcripts...")
dataset["train"] = dataset["train"].map(lambda x: {"text": clean_medical_text(x["text"])})
dataset["validation"] = dataset["validation"].map(lambda x: {"text": clean_medical_text(x["text"])})

Cleaning text transcripts...


Map:   0%|          | 0/160 [00:00<?, ? examples/s]

Map:   0%|          | 0/40 [00:00<?, ? examples/s]

In [25]:
# TOKENIZATION
model_ckpt = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

def tokenize_function(examples):
    # Max length 256 preserves memory; stretch to 512 if transcriptions are very long
    return tokenizer(examples["text"], truncation=True, max_length=512, padding="max_length")

print("Tokenizing data...")
tokenized_datasets = dataset.map(tokenize_function, batched=True)

Tokenizing data...


Map:   0%|          | 0/160 [00:00<?, ? examples/s]

Map:   0%|          | 0/40 [00:00<?, ? examples/s]

In [26]:
print(tokenized_datasets)

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 160
    })
    validation: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 40
    })
})


In [29]:
# MODEL & METRICS SETUP
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        # Extract labels from the input batch
        labels = inputs.get("labels")

        # Forward pass through DistilBERT
        outputs = model(**inputs)
        logits = outputs.get("logits")

        # Calculate custom weighted loss
        loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights_tensor)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss

model = AutoModelForSequenceClassification.from_pretrained(
    model_ckpt,
    num_labels=num_classes,
    id2label=id2label,
    label2id=label2id
).to(device)

accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    return accuracy_metric.compute(predictions=preds, references=labels)

# CONFIGURATION & TRAINING
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3, # Set to 3 or 4 epochs for text classification
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    fp16=torch.cuda.is_available(), # Memory optimization for Colab GPU
    report_to="none"
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

print("Starting training loop...")
trainer.train()

# SAVE THE OUTPUT MODEL
trainer.save_model("./my_fine_tuned_distilbert")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Starting training loop...


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.681779,0.425000
2,No log,0.629047,0.775000
3,No log,0.621348,0.750000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [31]:
# LOAD THE FINE-TUNED MODEL
classifier = pipeline(
    "text-classification",
    model="./my_fine_tuned_distilbert",
    tokenizer="./my_fine_tuned_distilbert",
    device=0 if torch.cuda.is_available() else -1
)

sample_transcript = [
    """CHIEF COMPLAINT:,  Foul-smelling urine and stomach pain after meals.,HISTORY OF PRESENT ILLNESS:,  Stomach pain with most meals x one and a half years and urinary symptoms for same amount of time.  She was prescribed Reglan, Prilosec, Pepcid, and Carafate at ED for her GI symptoms and Bactrim for UTI.  This visit was in July 2010.,REVIEW OF SYSTEMS:,  HEENT:  No headaches.  No visual disturbances, no eye irritation.  No nose drainage or allergic symptoms.  No sore throat or masses.  Respiratory:  No shortness of breath.  No cough or wheeze.  No pain.  Cardiac:  No palpitations or pain.  Gastrointestinal:  Pain and cramping.  Denies nausea, vomiting, or diarrhea.  Has some regurgitation with gas after meals.  Genitourinary:  "Smelly" urine.  Musculoskeletal:  No swelling, pain, or numbness.,MEDICATION ALLERGIES:,  No known drug allergies.,PHYSICAL EXAMINATION:,General:  Unremarkable.,HEENT:  PERRLA.  Gaze conjugate.,Neck:  No nodes.  No thyromegaly.  No masses.,Lungs:  Clear.,Heart:  Regular rate without murmur.,Abdomen:  Soft, without organomegaly, without guarding or tenderness.,Back:  Straight.  No paraspinal spasm.,Extremities:  Full range of motion.  No edema.,Neurologic:  Cranial nerves II-XII intact.  Deep tendon reflexes 2+ bilaterally.,Skin:  Unremarkable.,LABORATORY STUDIES:,  Urinalysis was done, which showed blood due to her period and moderate leukocytes.,ASSESSMENT:,1.  UTI.,2.  GERD.,3.  Dysphagia.,4.  Contraception consult.,PLAN:,1.  Cipro 500 mg b.i.d. x five days.  Ordered BMP, CBC, and urinalysis with microscopy.,2.  Omeprazole 20 mg daily and famotidine 20 mg b.i.d.,3.  Prescriptions same as #2.  Also referred her for a barium swallow series to rule out a stricture.,4.  Ortho Tri-Cyclen Lo.,""",
    """There is normal and symmetrical filling of the caliceal system.  Subsequent films demonstrate that the kidneys are of normal size and contour bilaterally.  The caliceal system and ureters are in their usual position and show no signs of obstruction or intraluminal defects. The postvoid films demonstrate normal emptying of the collecting system, including the urinary bladder.,IMPRESSION:,  Negative intravenous urogram.,""",
    """PROCEDURE: , Urgent cardiac catheterization with coronary angiogram.,PROCEDURE IN DETAIL: , The patient was brought urgently to the cardiac cath lab from the emergency room with the patient being intubated with an abnormal EKG and a cardiac arrest.  The right groin was prepped and draped in usual manner.  Under 2% lidocaine anesthesia, the right femoral artery was entered.  A 6-French sheath was placed.  The patient was already on anticoagulation.  Selective coronary angiograms were then performed using a left and a 3DRC catheter.  The catheters were reviewed.  The catheters were then removed and an Angio-Seal was placed.  There was some hematoma at the cath site.,RESULTS,1.  The left main was free of disease.,2.  The left anterior descending and its branches were free of disease.,3.  The circumflex was free of disease.,4.  The right coronary artery was free of disease.  There was no gradient across the aortic valve.,IMPRESSION: , Normal coronary angiogram.,""",
    """ADMITTING DIAGNOSES:,  Hiatal hernia, gastroesophageal reflux disease reflux.,DISCHARGE DIAGNOSES:,  Hiatal hernia, gastroesophageal reflux disease reflux.,SECONDARY DIAGNOSIS: , Postoperative ileus.,PROCEDURES DONE: , Hiatal hernia repair and Nissen fundoplication revision.,BRIEF HISTORY: , The patient is an 18-year-old male who has had a history of a Nissen fundoplication performed six years ago for gastric reflux.  Approximately one year ago, he was involved in a motor vehicle accident and CT scan at that time showed that he had a hiatal hernia.  Over the past year, this has caused him an increasing number of problems, including chest pain when he eats, and shortness of breath after large meals.  He is also having reflux symptoms again.  He presents to us for repair of the hiatal hernia and revision of the Nissen fundoplication.,HOSPITAL COURSE: , Mr. A was admitted to the adolescent floor by Brenner Children's Hospital after his procedure.  He was stable at that time.  He did complain of some nausea.  However, he did not have any vomiting at that time.  He had an NG tube in and was n.p.o.  He also had a PCA for pain management as well as Toradol.  On postoperative day #1, he complained of not being able to urinate, so a Foley catheter was placed.  Over the next several days, his hospital course proceeded as follows.  He continued to complain of some nausea; however, he did not ever have any vomiting.  Eventually, the Foley catheter was discontinued and he had excellent urine output without any complications.  He ambulated frequently.  He remained n.p.o. for three days.  He also had the NG tube in during that time.  On postoperative day #4, he began to have some flatus, and the NG tube was discontinued.  He was advanced to a liquid diet and tolerated this without any complications.  At this time, he was still using the PCA for pain control.  However, he was using it much less frequently than on days #1 and #2 postoperatively.  After tolerating the full liquid diet without any complications, he was advanced to a soft diet and his pain medications were transitioned to p.o. medications rather than the PCA.  The PCA was discontinued.  He tolerated the soft diet without any complications and continued to have flatus frequently.  On postoperative day #6, it was determined that he was stable for discharge to home as he was taking p.o. without any complications.  His pain was well controlled with p.o. pain medications.  He was passing gas frequently, had excellent urine output, and was ambulating frequently without any issues.,DISCHARGE CONDITION:,  Stable.,DISPOSITION: , Discharged to home.,DISCHARGE INSTRUCTIONS: , The patient was discharged to home with instructions for maintaining a soft diet.  It was also recommended that he does not drink any soda postoperatively.  He is instructed to keep his incision site clean and dry and it was also recommended that he avoid any heavy lifting.  He will be able to attend school when it starts in a few weeks.  However, he is not going to be able to play football in the near future.  He was given prescription for pain medication upon discharge.  He is instructed to contact Pediatric Surgery if he has any fevers, any nausea and vomiting, any chest pain, any constipation, or any other concerns."""[:512],
    """PREOPERATIVE DIAGNOSIS:,  A 60% total body surface area flame burns, status post multiple prior excisions and staged graftings.,POSTOPERATIVE DIAGNOSIS:,  A 60% total body surface area flame burns, status post multiple prior excisions and staged graftings.,PROCEDURES PERFORMED:,1.  Epidermal autograft on Integra to the back (3520 cm2).,2.  Application of allograft to areas of the lost Integra, not grafted on the back (970 cm2).,ANESTHESIA: , General endotracheal.,ESTIMATED BLOOD LOSS:,  Approximately 50 cc.,BLOOD PRODUCTS RECEIVED:,  One unit of packed red blood cells.,COMPLICATIONS: , None.,INDICATIONS: , The patient is a 26-year-old male, who sustained a 60% total body surface area flame burn involving the head, face, neck, chest, abdomen, back, bilateral upper extremities, hands, and bilateral lower extremities.  He has previously undergone total burn excision with placement of Integra and an initial round of epidermal autografting to the bilateral upper extremities and hands.  His donor sites have healed particularly over his buttocks and he returns for a second round of epidermal autografting over the Integra on his back utilizing the buttock donor sites, the extent they will provide coverage.,OPERATIVE FINDINGS:,1.  Variable take of Integra, particularly centrally and inferiorly on the back.  A fair amount of lost Integra over the upper back and shoulders.,2.  No evidence of infection.,3.  Healthy viable wound beds prior to grafting.,PROCEDURE IN DETAIL:,  The patient was brought to the operating room and positioned supine.  General endotracheal anesthesia was uneventfully induced and an appropriate time out was performed.  He was then repositioned prone and perioperative IV antibiotics were administered.  He was prepped and draped in the usual sterile manner.  All staples were removed from the Integra and the adherent areas of Silastic were removed.  The entire wound bed was further prepped with scrub brushes and more Betadine followed by a sulfamylon solution.  Hemostasis of the wound bed was ensured using epinephrine-soaked Telfa pads.  Following dermal tumescence of the buttocks, epidermal autografts were harvested 8 one-thousandths of an inch using the air Zimmer dermatome.  These grafts were passed to the back table where they were meshed 3:1.  The donor sites were hemostased using epinephrine-soaked Telfa and lap pads.  Once all the  grafts were meshed, we brought them back up onto the field, positioned them over the wounds beginning inferiorly and moving cephalad where we had best areas of Integra engraftment.  We were happy with the lie of the grafts and they were stapled into place.  The grafts were then overlaid with Conformant 2, which was also stapled into place.  Utilizing all of his buttocks skin, we did not have enough to cover his entire back, so we elected to apply allograft to the cephalad and a few areas on his flanks where we had had poor Integra engraftment.  Allograft was thawed and meshed 1:1.  It was then brought up onto the field, trimmed to fit and stapled into place over the wound.  Once the entirety of the posterior wounds on his back were covered out with epidermal autograft or allograft sulfamylon soaked dressings were applied.  Donor sites on his buttocks were dressed in Acticoat and secured with staples.  He was then repositioned supine and extubated in the operating room having tolerated the procedure without any apparent complications.  He was transported to PACU in stable condition."""[:512]
]

# Run prediction
results = classifier(sample_transcript)

# Print the result of prediction
for text, result in zip(sample_transcript, results):
    print(f"\nTranscript Sample: {text[:80]}...")
    print(f"Predicted Specialty: {result['label']} (Confidence: {result['score']:.4f})")


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


Transcript Sample: CHIEF COMPLAINT:,  Foul-smelling urine and stomach pain after meals.,HISTORY OF ...
Predicted Specialty: Not Urology (Confidence: 0.5259)

Transcript Sample: There is normal and symmetrical filling of the caliceal system.  Subsequent film...
Predicted Specialty: Not Urology (Confidence: 0.5375)

Transcript Sample: PROCEDURE: , Urgent cardiac catheterization with coronary angiogram.,PROCEDURE I...
Predicted Specialty: Not Urology (Confidence: 0.5749)

Transcript Sample: ADMITTING DIAGNOSES:,  Hiatal hernia, gastroesophageal reflux disease reflux.,DI...
Predicted Specialty: Urology (Confidence: 0.5753)

Transcript Sample: PREOPERATIVE DIAGNOSIS:,  A 60% total body surface area flame burns, status post...
Predicted Specialty: Not Urology (Confidence: 0.5590)
